[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/03.%20The%20Manual%20Reefer%20Monitoring%20Problem/P3-Tier-1_executed.ipynb)

---


# 3. The Manual Reefer Monitoring Problem

## Tier 1: The Pen-and-Paper Method (Mathematical Formulation)

This notebook studies a small assignment problem: assign each refrigerated container to exactly one inspector while respecting inspector capacity and minimizing a weighted objective based on inspection time, safety risk, and a priority-weighted term. The solution method is exhaustive enumeration: it evaluates every possible assignment and selects one with the smallest objective among those that satisfy the capacity limits.

**Inputs:** container priorities, stack levels, locations, inspection intervals, inspector capacities and experience levels, and container–inspector inspection times and safety-risk scores.

**Outputs:** an optimal feasible assignment for the stated model, its objective value, workload and risk summaries, visualizations, and a sensitivity analysis of the objective weights.

### Model assumptions and scope

- Each container is assigned to exactly one inspector for one inspection cycle.
- Each inspector's total inspection time must not exceed that inspector's capacity.
- Inspection time and safety risk depend on the container–inspector pair and are supplied as input data.
- The model assigns containers but does not choose an inspection order or start time. It therefore does not enforce inspection intervals as scheduling time windows.
- Container and inspector coordinates are shown in a location plot, but travel distance and travel time do not enter the objective or capacity calculation.
- The priority-weighted objective term is a constant for a given problem instance. It is included in the reported objective, but it cannot influence which assignment is selected. Consequently, this formulation does not prioritize the order or timing of inspections.
- The example uses small, illustrative data. The exhaustive method is exact for this assignment model, but its number of candidate assignments grows exponentially with the number of containers.

In [ ]:
# These libraries support the data model, exhaustive enumeration, tables, and plots.
import itertools
from dataclasses import dataclass
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8")

## Approach and mathematical model

The assignment decision answers one question for every container: which inspector will perform its inspection? The exhaustive search makes this decision for all containers in turn, rejects assignments that exceed an inspector's capacity, and compares the objective values of the remaining assignments. This is an exact search for the model defined below; it is not a mixed-integer programming solver.

### Sets, data, and decision variables

Let $C$ be the set of containers and $I$ the set of inspectors. For each container $c \in C$ and inspector $i \in I$, let $t_{ci}$ be the inspection time in minutes and $r_{ci}$ the safety-risk score. Let $p_c$ be the priority of container $c$, and let $K_i$ be inspector $i$'s capacity in minutes. The binary decision variable $x_{ci}$ equals 1 if inspector $i$ is assigned to container $c$, and 0 otherwise.

The objective combines total inspection time, total safety risk, and a priority-weighted constant:

$$
\min_{x}\quad
Z = \alpha \sum_{c \in C}\sum_{i \in I} t_{ci}x_{ci}
+ \beta \sum_{c \in C}\sum_{i \in I} r_{ci}x_{ci}
+ \gamma \sum_{c \in C} 0.1p_c.
$$

Here, $\alpha$, $\beta$, and $\gamma$ are the objective weights. The last term has no assignment variable: because every container is assigned exactly once, it is the same for every feasible assignment in a fixed instance. It affects the reported objective value but not the minimizing assignment.

Every container must have exactly one assigned inspector, and no inspector may exceed capacity:

$$
\sum_{i \in I} x_{ci}=1 \quad (c\in C),
\qquad
\sum_{c \in C}t_{ci}x_{ci}\leq K_i \quad (i\in I),
\qquad
x_{ci}\in\{0,1\} \quad (c\in C,\ i\in I).
$$

The first constraint is represented by generating one inspector choice for each container. The second is checked by accumulating each inspector's assigned inspection times. The binary condition is inherent in the enumeration: each choice is either to assign a container to an inspector or not to do so.

### Exhaustive-search logic

With $|C|$ containers and $|I|$ inspectors, there are $|I|^{|C|}$ possible assignments. The algorithm examines these assignments, discards capacity-infeasible ones, and retains a feasible assignment with minimum objective. Its worst-case running time is exponential in the number of containers; the small example makes every candidate assignment practical to inspect. If multiple assignments attain the same minimum, the implementation returns the first one encountered in the enumeration order.

In [ ]:
@dataclass
class Container:
    """Data describing a refrigerated container in the assignment instance."""

    id: int
    priority: int
    stack_level: int
    location_x: float
    location_y: float
    max_inspection_interval: int  # Hours; descriptive input, not scheduled here.


@dataclass
class Inspector:
    """Data describing an inspector's capacity, experience, and location."""

    id: int
    capacity_minutes: int
    experience_level: int
    current_location_x: float
    current_location_y: float


@dataclass
class InspectionTime:
    """Inspection time and safety risk for one container–inspector pair."""

    container_id: int
    inspector_id: int
    time_minutes: int
    safety_risk_score: float

### Interpreting the model's objectives

The objective weights express the relative importance assigned to inspection time and safety risk. They do not create hard safety requirements: the model has no constraint requiring high-stack containers to go to experienced inspectors. A lower safety-risk score can make such an assignment preferable when the objective weights and capacity limits support it, but it is not guaranteed by a separate rule.

The priority term is included to make the objective components explicit, but it is a fixed instance-level contribution rather than an assignment-dependent delay cost. A true scheduling model would need decisions about inspection order or timing to represent delays and enforce maximum inspection intervals; those decisions are outside this assignment model.

In [ ]:
def create_concrete_example():
    """Create the illustrative container, inspector, and pairwise data."""

    containers = [
        Container(1, 10, 1, 10, 20, 2),
        Container(2, 5, 2, 30, 40, 3),
        Container(3, 1, 1, 50, 30, 4),
        Container(4, 10, 3, 20, 60, 2),
        Container(5, 5, 1, 40, 10, 3),
        Container(6, 1, 2, 60, 50, 4),
    ]

    inspectors = [
        Inspector(1, 120, 2, 0, 0),
        Inspector(2, 120, 3, 0, 0),
    ]

    # Pairwise times and risks are input data for the assignment model.
    times_by_container = {
        1: {1: 2, 2: 3},
        2: {1: 5, 2: 4},
        3: {1: 2, 2: 2},
        4: {1: 8, 2: 6},
        5: {1: 2, 2: 3},
        6: {1: 5, 2: 4},
    }

    inspection_times = {}
    for container in containers:
        for inspector in inspectors:
            time_minutes = times_by_container[container.id][inspector.id]
            safety_risk = (
                container.stack_level
                * (4 - inspector.experience_level)
                * 0.5
            )
            inspection_times[(container.id, inspector.id)] = InspectionTime(
                container.id,
                inspector.id,
                time_minutes,
                safety_risk,
            )

    return containers, inspectors, inspection_times


containers, inspectors, inspection_times = create_concrete_example()

print("PROBLEM INSTANCE")
print(f"\nContainers ({len(containers)}):")
for container in containers:
    print(
        f"  C{container.id}: priority={container.priority}, "
        f"stack level={container.stack_level}, "
        f"location=({container.location_x}, {container.location_y}), "
        f"maximum inspection interval={container.max_inspection_interval} h"
    )

print(f"\nInspectors ({len(inspectors)}):")
for inspector in inspectors:
    print(
        f"  I{inspector.id}: capacity={inspector.capacity_minutes} min, "
        f"experience level={inspector.experience_level}, "
        f"location=({inspector.current_location_x}, "
        f"{inspector.current_location_y})"
    )

print("\nInspection times and safety-risk scores:")
for (container_id, inspector_id), inspection in inspection_times.items():
    print(
        f"  C{container_id}–I{inspector_id}: "
        f"{inspection.time_minutes} min, "
        f"risk={inspection.safety_risk_score:.1f}"
    )

### Reading the instance data

The `InspectionTime` record supplies the time and risk score for one particular container–inspector pair. These values are not recalculated by the optimizer; they are the input coefficients $t_{ci}$ and $r_{ci}$ in the mathematical model. The example's safety-risk formula makes risk increase with stack level and decrease with inspector experience. Capacity is measured in minutes, so inspection times and capacity limits use the same unit.

In [ ]:
class ReeferMonitoringOptimizer:
    """Exhaustive assignment optimizer for the stated reefer-monitoring model."""

    def __init__(self, containers, inspectors, inspection_times):
        self.containers = containers
        self.inspectors = inspectors
        self.inspection_times = inspection_times

        self.alpha = 1.0
        self.beta = 0.5
        self.gamma = 2.0

        self.container_by_id = {container.id: container for container in containers}
        self.inspector_by_id = {inspector.id: inspector for inspector in inspectors}

        if len(self.container_by_id) != len(containers):
            raise ValueError("Container IDs must be unique.")
        if len(self.inspector_by_id) != len(inspectors):
            raise ValueError("Inspector IDs must be unique.")

        required_pairs = {
            (container.id, inspector.id)
            for container in containers
            for inspector in inspectors
        }
        missing_pairs = required_pairs.difference(inspection_times)
        if missing_pairs:
            raise ValueError(f"Missing inspection data for pairs: {sorted(missing_pairs)}")

    def calculate_distance(self, x1, y1, x2, y2):
        """Return Euclidean distance in the coordinate system's units."""
        return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

    def solve_exhaustive_search(self):
        """Enumerate assignments and return a minimum-objective feasible one."""
        best_solution = None
        best_objective = float("inf")

        container_ids = [container.id for container in self.containers]
        inspector_ids = [inspector.id for inspector in self.inspectors]

        # Each tuple makes exactly one inspector choice for every container.
        for assignment in itertools.product(inspector_ids, repeat=len(container_ids)):
            inspector_loads = {inspector_id: 0 for inspector_id in inspector_ids}
            feasible = True

            for container_id, inspector_id in zip(container_ids, assignment):
                inspection = self.inspection_times[(container_id, inspector_id)]
                inspector_loads[inspector_id] += inspection.time_minutes
                capacity = self.inspector_by_id[inspector_id].capacity_minutes

                if inspector_loads[inspector_id] > capacity:
                    feasible = False
                    break

            if not feasible:
                continue

            total_time = 0
            total_safety_risk = 0.0
            for container_id, inspector_id in zip(container_ids, assignment):
                inspection = self.inspection_times[(container_id, inspector_id)]
                total_time += inspection.time_minutes
                total_safety_risk += inspection.safety_risk_score

            priority_constant = sum(
                0.1 * container.priority for container in self.containers
            )
            objective = (
                self.alpha * total_time
                + self.beta * total_safety_risk
                + self.gamma * priority_constant
            )

            if objective < best_objective and not math.isclose(
                objective,
                best_objective,
                rel_tol=1e-12,
                abs_tol=1e-12,
            ):
                best_objective = objective
                best_solution = assignment

        return best_solution, best_objective

    def analyze_solution(self, assignment):
        """Summarize assignment decisions, workloads, time, and safety risk."""
        if assignment is None:
            return None
        if len(assignment) != len(self.containers):
            raise ValueError("The assignment must contain one choice per container.")

        analysis = {
            "assignments": [],
            "inspector_workloads": {
                inspector.id: 0 for inspector in self.inspectors
            },
            "total_time": 0,
            "total_safety_risk": 0.0,
            "priority_distribution": {
                priority: 0
                for priority in sorted(
                    {container.priority for container in self.containers}
                )
            },
        }

        for container, inspector_id in zip(self.containers, assignment):
            if inspector_id not in self.inspector_by_id:
                raise ValueError(f"Unknown inspector ID: {inspector_id}")

            inspection = self.inspection_times[(container.id, inspector_id)]
            analysis["assignments"].append(
                {
                    "container_id": container.id,
                    "inspector_id": inspector_id,
                    "priority": container.priority,
                    "stack_level": container.stack_level,
                    "inspection_time": inspection.time_minutes,
                    "safety_risk": inspection.safety_risk_score,
                }
            )
            analysis["inspector_workloads"][inspector_id] += inspection.time_minutes
            analysis["total_time"] += inspection.time_minutes
            analysis["total_safety_risk"] += inspection.safety_risk_score
            analysis["priority_distribution"][container.priority] += 1

        return analysis

### How the search implements the mathematics

For six containers and two inspectors, `itertools.product` generates $2^6$ inspector-choice tuples. In each tuple, the search accumulates inspection time by inspector and discards the tuple as soon as a capacity limit is exceeded. For each remaining tuple, it calculates the three objective components and compares the total with the best feasible value found so far.

`analyze_solution` independently builds a readable summary from the selected assignment. It does not choose or alter the assignment; it connects the decision tuple returned by the search to container-level results and inspector workloads.

In [ ]:
optimizer = ReeferMonitoringOptimizer(containers, inspectors, inspection_times)

print("EXHAUSTIVE ASSIGNMENT SEARCH")
print("\nEvaluating feasible inspector assignments...")

optimal_assignment, optimal_objective = optimizer.solve_exhaustive_search()

if optimal_assignment is not None:
    analysis = optimizer.analyze_solution(optimal_assignment)

    print("\nMinimum-objective feasible assignment found.")
    print(f"Objective value: {optimal_objective:.2f}")

    print("\nASSIGNMENTS")
    for assignment in analysis["assignments"]:
        print(
            f"Container {assignment['container_id']} -> "
            f"Inspector {assignment['inspector_id']}: "
            f"time={assignment['inspection_time']} min, "
            f"priority={assignment['priority']}, "
            f"stack level={assignment['stack_level']}, "
            f"risk={assignment['safety_risk']:.1f}"
        )

    print("\nINSPECTOR WORKLOADS")
    for inspector_id, workload in analysis["inspector_workloads"].items():
        capacity = optimizer.inspector_by_id[inspector_id].capacity_minutes
        utilization = 100 * workload / capacity if capacity else 0.0
        print(
            f"Inspector {inspector_id}: {workload} / {capacity} min "
            f"({utilization:.1f}% of capacity)"
        )

    print("\nSOLUTION SUMMARY")
    print(f"Total inspection time: {analysis['total_time']} min")
    print(f"Total safety-risk score: {analysis['total_safety_risk']:.1f}")
    print(f"Priority counts: {analysis['priority_distribution']}")
else:
    print("No feasible assignment exists for this instance.")

### Interpreting the solution

The reported assignment is optimal with respect to the stated objective and capacity constraints because every possible assignment has been considered. The objective value includes the fixed priority contribution as well as the time and risk contributions. Workload utilization is each inspector's assigned inspection time divided by that inspector's capacity; it is a descriptive measure, not a separate balancing objective.

Optimality applies only to this model and its input data. It does not establish that inspection intervals are met, that travel is minimized, or that inspection order is safe, because those decisions and constraints are not represented.

### Visualizing the assignment

The location panel shows where containers and inspectors are located and uses color to identify the assigned inspector. Marker size represents container priority. The other panels summarize workload relative to capacity, the priority composition of the instance, and the safety-risk scores associated with the selected container–inspector pairs. The location plot is descriptive: it does not imply that travel distance is part of the optimization.

In [ ]:
def create_solution_visualization(analysis, containers, inspectors):
    """Plot the assignment and its workload, priority, and risk summaries."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    ax1, ax2, ax3, ax4 = axes.flat
    fig.suptitle(
        "Manual Reefer Monitoring: Assignment Model",
        fontsize=16,
        fontweight="bold",
    )

    inspector_colors = {
        inspector.id: color
        for inspector, color in zip(inspectors, ["#1f77b4", "#ff7f0e"])
    }
    assignment_by_container = {
        item["container_id"]: item["inspector_id"]
        for item in analysis["assignments"]
    }

    for inspector in inspectors:
        assigned_containers = [
            container
            for container in containers
            if assignment_by_container[container.id] == inspector.id
        ]
        ax1.scatter(
            [container.location_x for container in assigned_containers],
            [container.location_y for container in assigned_containers],
            c=inspector_colors[inspector.id],
            s=[100 + container.priority * 20 for container in assigned_containers],
            alpha=0.75,
            edgecolors="black",
            linewidths=1,
            label=f"Inspector {inspector.id} assignments",
        )
        ax1.scatter(
            inspector.current_location_x,
            inspector.current_location_y,
            c=inspector_colors[inspector.id],
            s=180,
            marker="^",
            edgecolors="black",
            linewidths=1,
            label=f"Inspector {inspector.id} starting location",
        )

    for container in containers:
        ax1.annotate(
            f"C{container.id}",
            (container.location_x, container.location_y),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
        )
    for inspector in inspectors:
        ax1.annotate(
            f"I{inspector.id}",
            (inspector.current_location_x, inspector.current_location_y),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
        )

    ax1.set_xlabel("X coordinate (coordinate units)")
    ax1.set_ylabel("Y coordinate (coordinate units)")
    ax1.set_title("Container Locations and Assignments")
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=8)

    inspector_ids = list(analysis["inspector_workloads"])
    workloads = [analysis["inspector_workloads"][i] for i in inspector_ids]
    capacity_by_id = {inspector.id: inspector.capacity_minutes for inspector in inspectors}
    capacities = [capacity_by_id[i] for i in inspector_ids]
    x_positions = np.arange(len(inspector_ids))
    width = 0.35

    ax2.bar(
        x_positions - width / 2,
        workloads,
        width,
        label="Assigned workload",
        color="#1f77b4",
        alpha=0.75,
    )
    ax2.bar(
        x_positions + width / 2,
        capacities,
        width,
        label="Capacity",
        color="#ff7f0e",
        alpha=0.75,
    )
    ax2.set_xlabel("Inspector")
    ax2.set_ylabel("Time (minutes)")
    ax2.set_title("Workload and Capacity")
    ax2.set_xticks(x_positions)
    ax2.set_xticklabels([f"Inspector {i}" for i in inspector_ids])
    ax2.legend()
    ax2.grid(True, axis="y", alpha=0.3)

    priorities = list(analysis["priority_distribution"])
    counts = [analysis["priority_distribution"][priority] for priority in priorities]
    priority_labels = [f"Priority {priority}" for priority in priorities]
    priority_colors = ["#2ca02c", "#ff7f0e", "#d62728"]
    ax3.pie(
        counts,
        labels=priority_labels,
        colors=priority_colors[:len(priorities)],
        autopct="%1.1f%%",
        startangle=90,
    )
    ax3.set_title("Container Priority Composition")

    safety_risks = [item["safety_risk"] for item in analysis["assignments"]]
    container_ids = [item["container_id"] for item in analysis["assignments"]]
    bars = ax4.bar(
        range(len(container_ids)),
        safety_risks,
        color="#9467bd",
        alpha=0.75,
    )
    ax4.set_xlabel("Container")
    ax4.set_ylabel("Safety-risk score")
    ax4.set_title("Risk by Assigned Container–Inspector Pair")
    ax4.set_xticks(range(len(container_ids)))
    ax4.set_xticklabels([f"C{container_id}" for container_id in container_ids])
    ax4.grid(True, axis="y", alpha=0.3)
    for bar, risk in zip(bars, safety_risks):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            f"{risk:.1f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    fig.tight_layout()
    plt.show()


if optimal_assignment is not None:
    create_solution_visualization(analysis, containers, inspectors)

### Verifying the selected assignment

Verification checks properties that follow directly from the model: every container appears once, all assigned inspectors are valid, inspector workloads stay within capacity, and the objective reconstructed from the assignment agrees with the search result. These checks validate the implementation; they do not add constraints or introduce another solution method.

In [ ]:
print("ASSIGNMENT VERIFICATION")

if optimal_assignment is None:
    print("No feasible assignment is available to verify.")
else:
    assigned_container_ids = [
        item["container_id"] for item in analysis["assignments"]
    ]
    assert len(assigned_container_ids) == len(containers)
    assert len(set(assigned_container_ids)) == len(containers)
    assert set(assigned_container_ids) == {container.id for container in containers}

    for inspector in inspectors:
        workload = analysis["inspector_workloads"][inspector.id]
        assert workload <= inspector.capacity_minutes

    reconstructed_objective = (
        optimizer.alpha * analysis["total_time"]
        + optimizer.beta * analysis["total_safety_risk"]
        + optimizer.gamma
        * sum(0.1 * container.priority for container in containers)
    )
    assert math.isclose(
        reconstructed_objective,
        optimal_objective,
        rel_tol=1e-12,
        abs_tol=1e-12,
    )

    print("Each container is assigned exactly once.")
    print("All inspector capacity limits are satisfied.")
    print("The reconstructed objective matches the search result.")

    high_stack_ids = {
        container.id for container in containers if container.stack_level >= 2
    }
    high_stack_assigned_to_expert = [
        item["container_id"]
        for item in analysis["assignments"]
        if item["container_id"] in high_stack_ids and item["inspector_id"] == 2
    ]
    print(f"High-stack containers: {sorted(high_stack_ids)}")
    print(
        "High-stack containers assigned to Inspector 2: "
        f"{sorted(high_stack_assigned_to_expert)}"
    )
    print(
        "This is a descriptive safety check; the model does not require "
        "high-stack containers to be assigned to an expert."
    )

## Why this tier is useful

Exhaustive enumeration provides a transparent exact benchmark for small assignment instances. Because every possible assignment is checked, the selected feasible assignment has minimum objective value for the specified data, weights, and capacity constraints. The explicit search also makes it possible to inspect why a candidate assignment is rejected or preferred.

The method's principal limitation is scale: the number of assignments is $|I|^{|C|}$. This growth makes exhaustive enumeration unsuitable for sufficiently large instances. The model also depends on complete, credible pairwise time and risk data, and it represents a static assignment rather than a changing schedule. Its results should therefore be interpreted as optimal for this defined model, not as a complete operational plan.

### Sensitivity analysis

Sensitivity analysis changes the objective weights while keeping the instance data and capacity constraints fixed. For each scenario, the exhaustive search is run again and the resulting assignment's objective components and inspector workloads are recorded. Comparing total time and risk across scenarios helps show how the weights affect the selected assignment. Since the priority-weighted term is constant for this instance, changing its weight changes the reported objective value but cannot by itself change the minimizing assignment.

In [ ]:
def sensitivity_analysis(optimizer):
    """Compare exhaustive-search solutions under alternative objective weights."""
    print("SENSITIVITY ANALYSIS")

    weight_scenarios = [
        {"alpha": 1.0, "beta": 0.5, "gamma": 2.0, "name": "Base case"},
        {"alpha": 2.0, "beta": 0.3, "gamma": 1.0, "name": "Efficiency focused"},
        {"alpha": 0.5, "beta": 2.0, "gamma": 1.0, "name": "Safety focused"},
        {"alpha": 0.5, "beta": 0.5, "gamma": 3.0, "name": "Priority-term weight focused"},
    ]

    original_weights = (optimizer.alpha, optimizer.beta, optimizer.gamma)
    results = []

    try:
        for scenario in weight_scenarios:
            optimizer.alpha = scenario["alpha"]
            optimizer.beta = scenario["beta"]
            optimizer.gamma = scenario["gamma"]

            assignment, objective = optimizer.solve_exhaustive_search()
            if assignment is None:
                continue

            scenario_analysis = optimizer.analyze_solution(assignment)
            results.append(
                {
                    "scenario": scenario["name"],
                    "objective": objective,
                    "total_time": scenario_analysis["total_time"],
                    "total_safety_risk": scenario_analysis["total_safety_risk"],
                    "inspector_1_load": scenario_analysis["inspector_workloads"].get(1, 0),
                    "inspector_2_load": scenario_analysis["inspector_workloads"].get(2, 0),
                }
            )
    finally:
        optimizer.alpha, optimizer.beta, optimizer.gamma = original_weights

    results_df = pd.DataFrame(results)
    print("\nResults by weight scenario:")
    print(results_df.to_string(index=False))

    if results_df.empty:
        print("\nNo feasible assignment was found in the tested scenarios.")
        return results_df

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    ax1, ax2, ax3, ax4 = axes.flat
    fig.suptitle(
        "Sensitivity to Objective-Function Weights",
        fontsize=14,
        fontweight="bold",
    )

    scenarios = results_df["scenario"]
    ax1.bar(scenarios, results_df["objective"], color="skyblue", alpha=0.8)
    ax1.set_ylabel("Objective value")
    ax1.set_title("Objective by Scenario")
    ax1.tick_params(axis="x", rotation=35)

    ax2.bar(scenarios, results_df["total_time"], color="lightcoral", alpha=0.8)
    ax2.set_ylabel("Inspection time (minutes)")
    ax2.set_title("Total Inspection Time")
    ax2.tick_params(axis="x", rotation=35)

    ax3.bar(
        scenarios,
        results_df["total_safety_risk"],
        color="lightgreen",
        alpha=0.8,
    )
    ax3.set_ylabel("Safety-risk score")
    ax3.set_title("Total Safety Risk")
    ax3.tick_params(axis="x", rotation=35)

    x_positions = np.arange(len(results_df))
    width = 0.35
    ax4.bar(
        x_positions - width / 2,
        results_df["inspector_1_load"],
        width,
        label="Inspector 1",
        color="#1f77b4",
        alpha=0.8,
    )
    ax4.bar(
        x_positions + width / 2,
        results_df["inspector_2_load"],
        width,
        label="Inspector 2",
        color="#ff7f0e",
        alpha=0.8,
    )
    ax4.set_ylabel("Workload (minutes)")
    ax4.set_title("Inspector Workloads")
    ax4.set_xticks(x_positions)
    ax4.set_xticklabels(scenarios, rotation=35, ha="right")
    ax4.legend()

    fig.tight_layout()
    plt.show()
    return results_df


sensitivity_results = sensitivity_analysis(optimizer)

## Summary

This notebook formulates reefer monitoring as a binary assignment problem and solves the small example by exhaustive enumeration. The search enforces one inspector per container and inspector capacity limits, then minimizes the weighted sum of inspection time, safety risk, and a fixed priority-weighted term. The returned assignment is optimal for those specific inputs and constraints.

The results can be inspected through container-level assignments, workload summaries, verification checks, visualizations, and weight scenarios. The model does not sequence inspections, enforce maximum inspection intervals, calculate travel costs, or require experienced inspectors for high-stack containers. Those limitations define what the reported optimality does—and does not—mean.

As a Tier 1 exact baseline, this method illustrates the relationship between decision variables, an objective function, feasibility constraints, and solution search. Its exponential number of candidate assignments makes it most appropriate for small instances and for understanding or benchmarking the stated model.